# Обучение собственной LLM

Проведём необходимые импорты.

In [1]:
import os
import re
import gc
from tqdm.auto import tqdm
from nltk.tokenize import sent_tokenize

import torch

from tokenizers import (
    decoders,
    models,
    pre_tokenizers,
    trainers,
    Tokenizer,
)

from transformers import (
    PreTrainedTokenizerFast, 
    DataCollatorForLanguageModeling,
    LlamaForCausalLM, 
    LlamaConfig,
    Trainer, 
    TrainingArguments)

from transformers import AutoTokenizer, AutoModelForCausalLM

from datasets import Dataset, load_dataset

from trl import SFTTrainer, SFTConfig

os.environ["WANDB_DISABLED"] = "true"

W0627 15:10:49.033000 15444 site-packages\torch\distributed\elastic\multiprocessing\redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.


Зададим пути

In [2]:
PATH_DATA = 'corpus'
PATH_TOKENIZER = 'models/tokenizer'

## Pretrain

На данном этапе решим упрощённую задачу: научим модель только структуре языка на узком домене русской литературы. Цель обучить модель продолжать фразы разумным текстом. В качестве данных будем использовать [Репозиторий RussianNovels](https://github.com/JoannaBy/RussianNovels/tree/master/corpus).

### Подготовка текстов.

Прежде чем создавать сам токенизатор нам потребуется предобработать текст, для этого напишем ряд функций, которые облегчат нашу работу.

Создадим функцию, которая вычитает все текстовые файлы и объединит их в один общий корпус.

In [3]:
def load_data(path: str) -> str:
    texts = []
    for filename in tqdm(os.listdir(path), desc='Чтение файлов'):
            with open(os.path.join(path, filename), encoding='utf-8') as f:
                text = f.read().strip()
                texts.append(text)     
    return '\n'.join(texts)

Напишем функцию, которая будет удалять предложения которые содержат символы не из кириллицы, повторяющуюся пунктуацию и лишние пробелы.

In [4]:
def clean_text(text: str) -> str:
    sentences = sent_tokenize(text)
    res = [] 
    pattern = r'^[\u0400-\u04FF\s\d.,!?\-:;"\'()$$$${}\—–«»]*$'
    for sentence in tqdm(sentences, desc='Очистка текста'):       
        if re.fullmatch(pattern, sentence):
            sentence = re.sub(r'([.]+)|([,]+)|([?]+)|([!]+)|([-]+)|(\s+)', 
                              lambda m: m.group(0)[0], 
                              sentence)
            sentence = re.sub(r'\s+', ' ', sentence)
            
            res.append(sentence)
    return ' '.join(res)

Напишем функцию, которая будет получать чанки длиной не более заданного контекста исходя из разбиения по предложениям.

In [5]:
def get_chunk(text: str, max_length: int = 256):
    sentences = sent_tokenize(text)
    current_chunk = sentences[0]
    current_len = len(current_chunk.split())
    res = []
    for i in tqdm(range(1, len(sentences)), desc='Разбиение на чанки'):
        new_len = current_len + len(sentences[i].split())
        if new_len <= max_length - 2:
            current_chunk = current_chunk + ' ' + sentences[i]
            current_len = new_len
        else:
            res.append(current_chunk)
            current_chunk = sentences[i]
            current_len = len(sentences[i].split())
    if res[-1] != current_chunk:
        res.append(current_chunk)
    return res

Теперь поулчим предобработанные и разбитые тексты

In [6]:
texts = load_data(PATH_DATA)
texts = clean_text(texts)
texts = get_chunk(texts)

Чтение файлов:   0%|          | 0/108 [00:00<?, ?it/s]

Очистка текста:   0%|          | 0/511832 [00:00<?, ?it/s]

Разбиение на чанки:   0%|          | 0/520233 [00:00<?, ?it/s]

In [7]:
print(len(texts))

25730


Посмотрим у первых 3 примеров первые 100 символов, чтобы убедиться, что всё отработало корректно 

In [8]:
for t in texts[:3]:
    print(t[:100])

Посвящается Любови Евгеньевне Белозерской Пошел мелкий снег и вдруг повалил хлопьями. Ветер завыл; с
Алексей, Елена, Тальберг и Анюта, выросшая в доме Турбиной, и Николка, оглушенный смертью, с вихром,
Но часы, по счастью, совершенно бессмертны, бессмертен и Саардамский Плотник, и голландский изразец,


Всё отработало корректно, можем приступать к созданию токенизатора.

### Создание токенизатора.

В качестве токенизатора выберем BPE.

Создадим токенизатор

In [9]:
tokenizer = Tokenizer(models.BPE())
tokenizer.pre_tokenizer = pre_tokenizers.Whitespace()
trainer = trainers.BpeTrainer(vocab_size=7000 - 4, special_tokens=["<|endoftext|>"], continuing_subword_prefix='##')
tokenizer.train_from_iterator(texts, trainer=trainer)

Посмотрим как наш токенизатор токенизирует строку.

In [10]:
test_text = 'Однажды, в студённую зимнюю пору'
encoding = tokenizer.encode(test_text)
print(encoding.tokens)


['Одна', '##жды', ',', 'в', 'студ', '##ё', '##нную', 'зим', '##нюю', 'пору']


Добавим указания декодеру как объединять разбитые слова. И проверим на примере выше, как всё отработает.

In [11]:
tokenizer.decoder = decoders.WordPiece(prefix='##')
tokenizer.decode(encoding.ids)

'Однажды, в студённую зимнюю пору'

Сохраним токенизатор

In [12]:
tokenizer = PreTrainedTokenizerFast(
    tokenizer_object=tokenizer,
    bos_token="<|beginftext|>",
    eos_token="<|endoftext|>",
    pad_token="<|PAD|>",
    unk_token="<|UNK|>",
)

In [13]:
print("Vocab size of tokenizer:", tokenizer.vocab_size)
print("Max token ID in tokenizer:", max(tokenizer.get_vocab().values()))

Vocab size of tokenizer: 6996
Max token ID in tokenizer: 6998


Сохраним токенизатор

In [14]:
tokenizer.save_pretrained(PATH_TOKENIZER)

('models/tokenizer\\tokenizer_config.json',
 'models/tokenizer\\special_tokens_map.json',
 'models/tokenizer\\tokenizer.json')

Создадим датасета

In [15]:
train_dataset = Dataset.from_dict({'text': texts})

In [16]:
def tokenize_function(examples):
    return tokenizer(examples['text'], max_length=512, truncation=True, padding=True)   

tokenized_dataset = train_dataset.map(
    tokenize_function,
    batched=True,
    remove_columns=train_dataset.column_names
)

Map:   0%|          | 0/25730 [00:00<?, ? examples/s]

Инициализируем модель со 140 миллионами параметров.

In [17]:
config = LlamaConfig(
    hidden_size=1024,
    intermediate_size=1536, 
    num_hidden_layers=16,
    num_attention_heads=16,
    num_key_value_heads=8,
    vocab_size=7000,
)

model = LlamaForCausalLM(config)
params = sum(p.numel() for p in model.parameters())
print(f"Параметров: {params:,} (~{params/1e6:.1f}M)")

Параметров: 140,198,912 (~140.2M)


Укажем аргументы для обучения модели.

In [18]:
train_args = TrainingArguments(
    output_dir='models/pretrain_llama',
    overwrite_output_dir=True,
    num_train_epochs=20,
    learning_rate=1e-4,
    weight_decay=0.01,
    per_device_train_batch_size=32,
    save_steps=500,
    save_total_limit=2,
    prediction_loss_only=True,
    remove_unused_columns=False,
    logging_dir='models/logs',
    logging_steps=500,
    torch_compile=True, 
    fp16=True,
    dataloader_pin_memory=True
)

Using the `WANDB_DISABLED` environment variable is deprecated and will be removed in v5. Use the --report_to flag to control the integrations used for logging result (for instance --report_to none).


Создадим функцию коллатор и экземпляр тренера

In [19]:
data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False
)

trainer = Trainer(
    model=model,
    args=train_args,
    data_collator=data_collator,
    train_dataset=tokenized_dataset
)

In [20]:
trainer.train()

Step,Training Loss
500,5.963200
1000,4.627000
1500,4.218900
2000,3.932600
2500,3.785800
3000,3.574600
3500,3.412700
4000,3.311200
4500,3.022800
5000,2.923300


TrainOutput(global_step=16100, training_loss=1.9446758594276001, metrics={'train_runtime': 10402.4453, 'train_samples_per_second': 49.469, 'train_steps_per_second': 1.548, 'total_flos': 2.103020768722944e+17, 'train_loss': 1.9446758594276001, 'epoch': 20.0})

Удалим тренера.

In [21]:
del trainer

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.synchronize()

Посмотрим результаты генерации модели на тестовых промтах.

In [22]:
test_prompts = [
    "Все мысли, которые имеют огромные последствия",
    "Сила войска зависит от его духа",
    "Мысль о том, что он принес страдания",
    "Человек сознает себя свободным",
    "Что бы ни случилось, я всегда буду",
    "Любовь мешает смерти",
    "Нет, жизнь не кончена",
    "Всякая мысль, даже самая простая",
    "Война не любезность, а самое гадкое дело",
    "Чтобы жить честно"
] 

In [23]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'

model.to(device)

LlamaForCausalLM(
  (model): LlamaModel(
    (embed_tokens): Embedding(7000, 1024)
    (layers): ModuleList(
      (0-15): 16 x LlamaDecoderLayer(
        (self_attn): LlamaAttention(
          (q_proj): Linear(in_features=1024, out_features=1024, bias=False)
          (k_proj): Linear(in_features=1024, out_features=512, bias=False)
          (v_proj): Linear(in_features=1024, out_features=512, bias=False)
          (o_proj): Linear(in_features=1024, out_features=1024, bias=False)
        )
        (mlp): LlamaMLP(
          (gate_proj): Linear(in_features=1024, out_features=1536, bias=False)
          (up_proj): Linear(in_features=1024, out_features=1536, bias=False)
          (down_proj): Linear(in_features=1536, out_features=1024, bias=False)
          (act_fn): SiLU()
        )
        (input_layernorm): LlamaRMSNorm((1024,), eps=1e-06)
        (post_attention_layernorm): LlamaRMSNorm((1024,), eps=1e-06)
      )
    )
    (norm): LlamaRMSNorm((1024,), eps=1e-06)
    (rotary_emb): L

In [24]:
tokenizer = AutoTokenizer.from_pretrained(os.path.abspath(PATH_TOKENIZER))

In [25]:
model.eval()
for prompt in test_prompts:
    inputs = tokenizer(prompt, return_tensors="pt")
    with torch.no_grad():
        outputs = model.generate(
            inputs.input_ids.to(model.device),
            max_new_tokens = 50,
            pad_token_id = tokenizer.pad_token_id,
            eos_token_id = tokenizer.eos_token_id,        
            temperature = 0.5,
            do_sample = True,
            top_p = 0.9,
            repetition_penalty = 1.1,
        )

    print(tokenizer.decode(outputs[0], skip_special_tokens=True))



Все мысли, которые имеют огромные последствия отряхлых шагом, с вечным и навсегда веселым лицом. Но и в эту сторону бывают минуты ужасный образ его, который оглушал голову старухе, были маленькие густые ряды. - Нет, они такие добрые,
Сила войска зависит от его духа в Европе и смерть ; а старушка, осталась в гостиной, не знаю, читал она ей и не думала возиться. Наконец по зале, перед стуком и ночью, вернулся доктор и с удовольствием спросил : – Я извиняюсь,
Мысль о том, что он принес страдания, что ему тяжело жить, и он больше ни с чем сам себе мне поручить, после его будущего. Я всегда теперь ошибаюсь и считаю над ним смеем ; я уверен, что оно будет еще самостоятельнее, то зато
Человек сознает себя свободным, - насмешливо говоря : - Да, это несправедливы. Тогда он встанет ненавидя и теряет свою любовь к людьми, сына, которой его раскаяние, боясь ее, ненавидит не страстные. Клим Иванович
Что бы ни случилось, я всегда буду, я пишу все меня друг друга, вот всё еще пять лет ; ведь что тогд

Генерация средняя иногда язык связный, но иногда ощущается лёгкий рандом. И явно какие-то проблемы с двоеточием и точкой с запятой. Они почему-то отделяются от слов.

Удалим модель, прежде чем переходить ко второй части.

In [26]:
del model

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.synchronize()

## Post-train SFT

Вторая часть будет посвящена тренировке SFT для этого возьмём более крупную предобученную модель Qwen2.5-0.5B. Работать будем со следующим датасетом https://huggingface.co/datasets/d0rj/alpaca-cleaned-ru

Загрузим датасет.

In [27]:
dataset = load_dataset("d0rj/alpaca-cleaned-ru")

README.md:   0%|          | 0.00/760 [00:00<?, ?B/s]

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


(…)-00000-of-00001-c503683bee003a5c.parquet:   0%|          | 0.00/36.6M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/51760 [00:00<?, ? examples/s]

In [28]:
dataset

DatasetDict({
    train: Dataset({
        features: ['input', 'instruction', 'output'],
        num_rows: 51760
    })
})

Посмотрим что из себя представляют признаки

In [29]:
print('input', dataset['train'][0]['input'])
print('instruction', dataset['train'][0]['instruction'])
print('output', dataset['train'][0]['output'])

input 
instruction Дайте три совета, как оставаться здоровым.
output 1. Соблюдайте сбалансированную и питательную диету. Убедитесь, что в ваш рацион входят разнообразные фрукты и овощи, нежирный белок, цельнозерновые продукты и полезные жиры. Это помогает обеспечить ваш организм необходимыми питательными веществами для оптимального функционирования и может помочь предотвратить хронические заболевания.

2. Занимайтесь регулярной физической активностью. Упражнения имеют решающее значение для поддержания крепких костей, мышц и здоровья сердечно-сосудистой системы. Старайтесь уделять не менее 150 минут умеренным аэробным упражнениям или 75 минут интенсивным упражнениям каждую неделю.

3. Высыпайтесь. Достаточное количество качественного сна имеет решающее значение для физического и психического благополучия. Он помогает регулировать настроение, улучшать когнитивные функции и поддерживает здоровый рост и иммунную функцию. Старайтесь спать 7-9 часов каждую ночь.


Напишем функцию для создания инструкций для ЛЛМ

In [30]:
def examples_to_messages(examples):
    data = {'messages': []}
    for example in tqdm(examples):
        data['messages'].append([
            {'role': 'system', 'content': 'Ты - полезный AI-ассистент'},
            {'role': 'user', 'content': example['instruction']},
            {'role': 'assistant', 'content': example['output']}
        ])
    return Dataset.from_dict(data)

ds = examples_to_messages(dataset['train'])

  0%|          | 0/51760 [00:00<?, ?it/s]

Загрузим модель и токенизатор.

In [31]:
model_name = "Qwen/Qwen2.5-0.5B"

tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
        model_name,
        device_map="auto"
    )

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/681 [00:00<?, ?B/s]

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


model.safetensors:   0%|          | 0.00/988M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/138 [00:00<?, ?B/s]

Токенизируем текст в датасете

In [32]:
ds = ds.map(lambda x: {'text': tokenizer.apply_chat_template(x['messages'], tokenize=False)})

Map:   0%|          | 0/51760 [00:00<?, ? examples/s]

Инициализируем конфиг для тренера и сам тренер.

In [ ]:
config = SFTConfig(
    output_dir="models/qwen2.5-0.5b_sft",
    overwrite_output_dir=True,
    learning_rate=1e-5,
    weight_decay=0.01,
    per_device_train_batch_size=1,
    max_seq_length=512,
    num_train_epochs=1,
    report_to='none',
    save_strategy='no',
    dataset_text_field = "text",
    logging_dir='models/logs',
    logging_steps=500,
    torch_compile=True, 
    fp16=True,
    dataloader_pin_memory=True
)

trainer = SFTTrainer(
    model,
    args=config,
    train_dataset=ds,
    processing_class=tokenizer,
)

Converting train dataset to ChatML:   0%|          | 0/51760 [00:00<?, ? examples/s]

Applying chat template to train dataset:   0%|          | 0/51760 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/51760 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/51760 [00:00<?, ? examples/s]

In [ ]:
trainer.train()

Step,Training Loss


Удалим тренер.

In [ ]:
del trainer

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.synchronize()

Протестируем модель на примерах ниже.

In [ ]:
questions_rus = [
    "сколько планет в нашей солнечной системе?",
    "расскажи стих",
    "когда собирать крыжовник?",
    "Как быстро выучить новый язык?"
  ] 

In [ ]:
model.eval()
    
for question in questions_rus:
    
    messages = [
        {"role": "system", "content": "Ты - полезный AI-ассистент."},
        {"role": "user", "content": question}
    ]
    
    input_text = tokenizer.apply_chat_template(
        messages, 
        tokenize=False, 
        add_generation_prompt=True
    )

    inputs = tokenizer(input_text, return_tensors="pt").to(model.device)
    
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=128,  
            temperature = 0.8,
            do_sample = True,
            top_p = 0.9,
            repetition_penalty = 1.1,
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )
    
    response = tokenizer.decode(outputs[0], skip_special_tokens=False)
    assistant_response = response.split("<|im_start|>assistant\n")[-1].split("<|im_end|>")[0]

    
    print(f'\nВопрос: {question}')
    print(f'Ответ: {assistant_response}')
    print('-' * 50)


Вопрос: сколько планет в нашей солнечной системе?
Ответ: Согласно Вселенной, в нашей Солнечной системе 8 планет. Это в основном делают это, потому что они образуются из комбинации материала, который сейчас находится под угрозой исчезновения или находится в процессе распада. Это включает в себя такие вещи, как газ, водяной пар, остатки древних звезд и стабильные наборы материалов. Некоторые из этих материалов могут упоминаться в космологии и астрономии, а некоторые
--------------------------------------------------

Вопрос: расскажи стих
Ответ: Поскольку ветерок дует и я чувствую себя один,
Недостаток, который я так сильно испытываю.
Лучше быть рядом с кем-то, чем с ним напрямую.
Вместо того, чтобы жить от него,
Что ты можешь сделать, когда ты устали,
Ибо тебе просто нужно время для отдыха.

Это было, это было, это было,
Один момент, один день, и все, что вы можете дать.
Спасибо за то, что ты здесь,
Для того
--------------------------------------------------

Вопрос: когда собирать кры

Нейросеть научилась корректно работать с инструкциями и правильно возвращать ответ. Она правильно ответила про количество планет в солнечной системе, стих похож на стих, хоть и без рифмы и даже в ответе про изучение языков идёт обсуждение на заданную тему, но с крыжовником вышел косяк. Язык приемлемый, однако местами возникают проблемы со связностью.

## Выводы

В данном проекте рассмотрены два первых этапа обучения ЛЛМ: 
- Pretrain, в ходе которого модель научилась генерировать более-менее связный текст, хотя иногда наблюдаются проблемы. 
- SFT. На данном этапе модель научилась работать с инструкциями и весьти диалог на русском языке. Модель, как правило, понимает суть вопроса и умеет отвечать на них, хотя иногда возникают небольшие проблемы с адекватностью модели.